# Elliptic-curve cryptography

Inspect the two phase measurements of the locked elliptic-curve example, then look at the fixed and windowed point additions they call.

Adapted from the Qualtran v0.7.0 ECC tutorial. Copyright 2023 Google LLC. Licensed under Apache-2.0.

In [ ]:
from collections import Counter

from IPython.display import Markdown, SVG, display
from sympy import symbols

from examples.algorithms.field_addition_catalog import field_signature_svg
from flagquantum.algorithms.primitives.elliptic_curve_double_phase import (
    EllipticCurveDoublePhase, elliptic_curve_double_phase_source_graph,
)
from flagquantum.algorithms.primitives.elliptic_curve_fixed_point import (
    EllipticCurveFixedPointAddition,
)
from flagquantum.algorithms.primitives.elliptic_curve_phase_estimation import (
    elliptic_curve_phase_signature, elliptic_curve_phase_source_graph,
)
from flagquantum.algorithms.primitives.elliptic_curve_point import ECPoint
from flagquantum.algorithms.primitives.elliptic_curve_window_addition import (
    EllipticCurveWindowAddition, windowed_point_addition_source_graph,
)
from flagquantum.compiler._component_trace.call_graph import symbolic_call_graph
from flagquantum.drawer.call_graph import (
    export_call_graph_dot, format_call_graph_markdown, render_call_graph_svg,
)
from flagquantum.drawer.register_graph import export_register_graph_dot
from flagquantum.drawer.register_score import RegisterStep, layout_register_score
from flagquantum.dynamic import DynamicCircuit

def show_calls(graph):
    dot = export_call_graph_dot(graph.root, graph.direct_calls)
    display(SVG(render_call_graph_svg(dot)))
    display(Markdown(format_call_graph_markdown(graph.root, graph.direct_calls, graph.leaf_counts)))

## Two phase estimations

For points `P` and `Q=[k]P`, translation by either point preserves the eigenstates of translation by `P`. The two phases encode an index and its product with `k` modulo the point order. This circuit exposes the two readouts. Any classical recovery also needs the point order and suitable measurement results.

The source prepares the raw coordinates of `P`, applies both estimations to the same data, then releases the coordinates dirty. The native binding makes the data and independent readout slots explicit. Its executable path uses windows and inherits their clean-domain conditions. A raw source word can encode a different starting point when the Montgomery factor differs from one.

[Source method](https://arxiv.org/abs/2306.08585), Figure 4(a).

In [ ]:
n, p, px, py, qx, qy = symbols("n p P_x P_y Q_x Q_y")
symbolic = EllipticCurveDoublePhase(n, ECPoint(px, py, p), ECPoint(qx, qy, p), 1)
display(SVG(field_signature_svg((("FindECCPrivateKey", symbolic.source_signature, {}),))))

In [ ]:
grouped = elliptic_curve_double_phase_source_graph()
assert grouped.leaf_counts == {"ECPhaseEstimateR": 2}
show_calls(grouped)

In [ ]:
steps = (
    RegisterStep("Initialize raw P.x", outputs=("x",)),
    RegisterStep("Initialize raw P.y", outputs=("y",)),
    RegisterStep("Phase at P", ("x", "y"), ("x", "y")),
    RegisterStep("Phase at Q", ("x", "y"), ("x", "y")),
    RegisterStep("Dirty release x", ("x",)),
    RegisterStep("Dirty release y", ("y",)),
)
score = layout_register_score({"x": 3, "y": 3}, (), steps)
display(SVG(render_call_graph_svg(export_register_graph_dot(score))))

In [ ]:
base, public = ECPoint(0, 1, 3, 1), ECPoint(1, 0, 3, 1)
assert 2 * base == public
operation = EllipticCurveDoublePhase(3, base, public, 3)
q, c = operation.workspace_requirements()
x, y, first_phase, second_phase = (tuple(range(i * 3, (i + 1) * 3)) for i in range(4))
circuit = DynamicCircuit(12 + q).append(operation.make_on(
    x=x, y=y, first_phase=first_phase, second_phase=second_phase,
    workspace=tuple(range(12, 12 + q)),
    classical_bits=tuple(range(c)),
    first_results=tuple(range(c, c + 3)), second_results=tuple(range(c + 3, c + 6)),
))
emitted = Counter(instruction.name for instruction in circuit.to_ir().instructions)
assert emitted["measure"] == 2692 and emitted["ccx"] == 38
{"bound_wires": circuit.n_qubits, "emitted_measurements": emitted["measure"]}

## Point phase estimation

Prepare `n` phase bits, apply point translations at successive powers of two, and read out in the original wire order with a forward QFT. Each grouped window carries a big-endian address. The scalar source child has a classical map and no gate decomposition.

In [ ]:
phase_symbolic = elliptic_curve_phase_signature(n)
phase_small = elliptic_curve_phase_signature(3)
display(SVG(field_signature_svg((
    ("ECPhaseEstimateR(n)", phase_symbolic, {}),
    ("ECPhaseEstimateR(3)", phase_small, {}),
))))

In [ ]:
phase_calls = elliptic_curve_phase_source_graph(n)
assert phase_calls.leaf_counts == {"ECAddR": n // 1, "MeasureQFT": 1}
show_calls(phase_calls)

## Fixed-point addition

Add a constant affine point under one scalar control. This source leaf declares a classical map on raw coordinates. Its gate decomposition is opaque.

In [ ]:
rx, ry = symbols("R_x R_y")
fixed_symbolic = EllipticCurveFixedPointAddition(n, ECPoint(rx, ry, p))
P = ECPoint(15, 13, 17)
fixed_small = EllipticCurveFixedPointAddition(5, P)
display(SVG(field_signature_svg((
    ("ECAddR(n)", fixed_symbolic.source_signature, {}),
    ("ECAddR(5)", fixed_small.source_signature, {}),
))))

In [ ]:
show_calls(symbolic_call_graph("ECAddR", {}))

In [ ]:
for j in range(1, 21):
    add = EllipticCurveFixedPointAddition(5, j * P)
    _, result_x, result_y = add.classical_values(1, P.x, P.y)
    assert (result_x, result_y) == (((j + 1) * P).x, ((j + 1) * P).y)
    print(f"+[{j:2d}] P  ->  ({result_x:2d}, {result_y:2d})")

## Windowed point addition

Load a point multiple and its tangent into Montgomery words, apply the six-stage addition, and erase the retained banks. The original example below has width 16, modulus 7, base point `(2,2)` with curve coefficient 3, and a four-bit address.

[Validation and inherited source boundaries](../../docs/development/ECC_DOUBLE_PHASE_FILE_AUDIT.md)

In [ ]:
window = EllipticCurveWindowAddition(16, ECPoint(2, 2, 7, 3), 4)
display(SVG(field_signature_svg((("ECWindowAddR(16,4)", window.source_signature, {}),))))
assert window.table.shape == (16,) and len(window.table.datasets) == 3

In [ ]:
window_calls = windowed_point_addition_source_graph()
assert sum(window_calls.leaf_counts.values()) == 3
show_calls(window_calls)